Imports


In [1]:
import sqlite3
import pandas as pd
from datetime import datetime
import os

DWH = sqlite3.connect("../../Database/DWH.db")
SDM = sqlite3.connect("../../Database/SDM2.db")


Logging

In [2]:
log_list = []

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

def log(source, table, action, key, expected, actual, status, note):
    log_list.append({
        "run_id": run_id,
        "source": source,
        "table": table,
        "action": action,
        "key": key,
        "expected": expected,
        "actual": actual,
        "status": status,
        "note": note
    })


Extract


In [3]:
df_klant = pd.read_sql_query("SELECT * FROM Fietsverkoop_Klant", SDM)
df_monteur = pd.read_sql_query("SELECT * FROM Fietsverkoop_Monteur", SDM)
df_fiets = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fiets", SDM)
df_fact = pd.read_sql_query("SELECT * FROM Fietsverkoop_Fiets_Verkoop", SDM)


Transform - clean data

In [4]:
df_klant_clean = pd.DataFrame({
    "klantnr": df_klant["klantnr"],
    "naam": df_klant["naam"],
    "woonplaats": df_klant["woonplaats"],
    "geslacht": df_klant["geslacht"],
    "geboortedatum": df_klant["geboortedatum"]
})


df_monteur_clean = pd.DataFrame({
    "monteurnr": df_monteur["monteurnr"],
    "naam": df_monteur["naam"],
    "uurloon": df_monteur["uurloon"],
    "filiaal_naam": df_monteur["filiaal"],   # rename
    "filiaal_provincie": None               # placeholder (or map if you have it)
})


df_fiets_clean = pd.DataFrame({
    "fietsnr": df_fiets["fietsnr"],
    "soort": df_fiets["soort"],
    "merk": df_fiets["merk"],
    "type": df_fiets["type"],
    "kleur": df_fiets["kleur"],
    "standaardprijs": df_fiets["standaardprijs"],
    "inkoopprijs": df_fiets["inkoopprijs"]
})






4. - •	Laat Python detecteren welke tabelrijen uit het Source Data Model (SDM) nog niet voorkomen in het Datawarehouse.

In [5]:
def detect_new_rows(sdm_df, dwh_df, key):
    return sdm_df[~sdm_df[key].isin(dwh_df[key])]


In [6]:
dwh_klant = pd.read_sql_query("SELECT klantnr FROM Dim_Klant", DWH)

new_klant = detect_new_rows(df_klant_clean, dwh_klant, "klantnr")

print("Nieuwe klant rijen:", len(new_klant))


dwh_monteur = pd.read_sql_query("SELECT monteurnr FROM Dim_Monteur", DWH)

new_monteur = detect_new_rows(df_monteur_clean, dwh_monteur, "monteurnr")

print("Nieuwe monteur rijen:", len(new_monteur))

dwh_fiets = pd.read_sql_query("SELECT fietsnr FROM Dim_Fiets", DWH)

new_fiets = detect_new_rows(df_fiets_clean, dwh_fiets, "fietsnr")

print("Nieuwe fiets rijen:", len(new_fiets))





Nieuwe klant rijen: 0
Nieuwe monteur rijen: 0
Nieuwe fiets rijen: 0


4.2 scd1 voorbeeld met monteur dim

In [7]:
def load_scd1(df, table, key):

    insert_count = 0
    update_count = 0

    for _, row in df.iterrows():

        existing = pd.read_sql_query(
            f"SELECT * FROM {table} WHERE {key} = ?",
            DWH,
            params=(row[key],)
        )

        if not existing.empty:
            # UPDATE
            columns = [c for c in df.columns if c != key]
            set_clause = ", ".join([f"{c}=?" for c in columns])
            values = [row[c] for c in columns]
            values.append(row[key])

            DWH.execute(
                f"UPDATE {table} SET {set_clause} WHERE {key} = ?",
                values
            )

            update_count += 1

        else:
            # INSERT
            pd.DataFrame([row]).to_sql(table, DWH, if_exists="append", index=False)

            insert_count += 1

    DWH.commit()

    # ✅ SUMMARY LOG
    log(
        "SDM → DWH",
        table,
        "SCD1 SUMMARY",
        "-",
        "Batch processed",
        f"Inserted: {insert_count}, Updated: {update_count}",
        "SUCCESS",
        "SCD1 load completed"
    )



In [8]:
load_scd1(df_monteur_clean, "Dim_Monteur", "monteurnr")


sdc 2 voorbeeld 1

In [9]:



def load_scd2(df, table, key, compare_cols):

    today = datetime.now().strftime("%Y-%m-%d")

    insert_count = 0
    update_count = 0
    nochange_count = 0

    for _, row in df.iterrows():

        existing = pd.read_sql_query(f"""
            SELECT * FROM {table}
            WHERE {key} = ? AND is_actueel = 1
        """, DWH, params=(row[key],))

        # 🟢 INSERT FIRST VERSION
        if existing.empty:

            new_row = row.to_dict()
            new_row["geldig_vanaf"] = today
            new_row["geldig_tot"] = None
            new_row["is_actueel"] = 1

            pd.DataFrame([new_row]).to_sql(table, DWH, if_exists="append", index=False)

            insert_count += 1

        else:
            old = existing.iloc[0]

            changed = any(
                str(old[col]) != str(row[col]) for col in compare_cols
            )

            # 🟡 NO CHANGE
            if not changed:
                nochange_count += 1

            # 🔴 CHANGE
            else:
                # close old
                DWH.execute(f"""
                    UPDATE {table}
                    SET geldig_tot = ?, is_actueel = 0
                    WHERE {key} = ? AND is_actueel = 1
                """, (today, row[key]))

                # insert new version
                new_row = row.to_dict()
                new_row["geldig_vanaf"] = today
                new_row["geldig_tot"] = None
                new_row["is_actueel"] = 1

                pd.DataFrame([new_row]).to_sql(table, DWH, if_exists="append", index=False)

                update_count += 1

    DWH.commit()

    # ✅ SUMMARY LOG
    log(
        "SDM → DWH",
        table,
        "SCD2 SUMMARY",
        "-",
        "Batch processed",
        f"Inserted: {insert_count}, Updated: {update_count}, No change: {nochange_count}",
        "SUCCESS",
        "SCD2 load completed"
    )


In [10]:
load_scd2(
    df_klant_clean,
    "Dim_Klant",
    "klantnr",
    ["naam", "woonplaats", "geslacht", "geboortedatum"]
)


In [11]:
load_scd2(
    df_fiets_clean,
    "Dim_Fiets",
    "fietsnr",
    ["soort", "merk", "type", "kleur", "standaardprijs", "inkoopprijs"]
)


Feit tabellen

In [12]:
df_fact = pd.read_sql_query(
    "SELECT * FROM Fietsverkoop_Fiets_Verkoop",
    SDM
)



In [13]:
existing_facts = pd.read_sql_query(
    "SELECT fiets_verkoop FROM Fact_Fiets_Verkoop",
    DWH
)

new_facts = df_fact[
    ~df_fact["fiets_verkoopnr"].isin(existing_facts["fiets_verkoop"])
]

print("Nieuwe fact rijen:", len(new_facts))


Nieuwe fact rijen: 0


In [14]:
def get_sk(table, key_name, value):

    sk_map = {
        "Dim_Klant": "klant_sk",
        "Dim_Fiets": "fiets_sk",
        "Dim_Monteur": "monteur_sk"
    }

    sk_column = sk_map[table]

    # check if SCD2 table
    has_scd2 = table in ["Dim_Klant", "Dim_Fiets"]

    if has_scd2:
        query = f"""
            SELECT {sk_column}
            FROM {table}
            WHERE {key_name} = ? AND is_actueel = 1
        """
    else:
        query = f"""
            SELECT {sk_column}
            FROM {table}
            WHERE {key_name} = ?
        """

    result = pd.read_sql_query(query, DWH, params=(value,))

    if len(result) == 0:
        return None

    return result.iloc[0, 0]


In [15]:
for _, row in new_facts.iterrows():

    # 🔑 surrogate keys (from dimensions)
    klant_sk = get_sk("Dim_Klant", "klantnr", row["klant"])
    fiets_sk = get_sk("Dim_Fiets", "fietsnr", row["fiets"])
    monteur_sk = get_sk("Dim_Monteur", "monteurnr", row["monteur"])

    # 💰 insert fact row
    DWH.execute("""
        INSERT INTO Fact_Fiets_Verkoop (
            fiets_verkoop,
            datum_sk,
            klant_sk,
            fiets_sk,
            monteur_sk,
            aantal,
            verkoopprijs,
            totale_prijs
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["fiets_verkoopnr"],
        row["datum"],   # (you can later improve this to Dim_Datum)
        klant_sk,
        fiets_sk,
        monteur_sk,
        row["aantal"],
        row["verkoopprijs"],
        row["aantal"] * row["verkoopprijs"]
    ))

DWH.commit()

fact_insert_count = len(new_facts)

log(
    "SDM → DWH",
    "Fact_Fiets_Verkoop",
    "FACT LOAD",
    "-",
    "New facts inserted",
    f"{fact_insert_count} rows inserted",
    "SUCCESS",
    "Incremental fact load"
)



save to logbook

In [16]:
file_path = "dwh_logbook3.xlsx"

new_log_df = pd.DataFrame(log_list)

if os.path.exists(file_path):
    existing_df = pd.read_excel(file_path)
    combined_df = pd.concat([existing_df, new_log_df], ignore_index=True)
else:
    combined_df = new_log_df

combined_df.to_excel(file_path, index=False)


Inlaadstrategie: Incremental Data Loading


In [17]:
print(df_klant_clean)
pd.read_sql_query("PRAGMA table_info(klant)", DWH)



    klantnr             naam  woonplaats geslacht geboortedatum
0         1       Jan Jansen   Amsterdam        M    1985-03-22
1         2   Sophie de Boer   Rotterdam        V    1990-07-11
2         3    Pieter Visser    Den Haag        M    1978-11-05
3         4        Emma Smit     Haarlem        V    1995-02-18
4         5       Tom Bakker      Leiden        M    1982-09-09
5         6      Lisa Meijer     Zaandam        V    1993-12-30
6         7    Bart de Vries       Delft        M    1976-06-06
7         8   Julia van Dijk       Hoorn        V    2000-01-15
8         9        Kevin Mol     Alkmaar        M    1989-08-01
9        10       Nina Groen    Schiedam        V    1991-05-25
10       11     Daan Willems     Haarlem        M    1987-10-12
11       12          Eva Vos  Zoetermeer        V    1996-04-03
12       13      Rik de Jong   Hilversum        M    1980-12-21
13       14     Mila Kuipers   Rotterdam        V    1998-09-17
14       15     Sam Reinders     Haarlem

,cid,name,type,notnull,dflt_value,pk


Testing!

test monteur

In [18]:
SDM.execute("""
UPDATE Fietsverkoop_Monteur
SET naam = 'monteur 123'
WHERE monteurnr = 1
""")
SDM.commit()


In [19]:
log("TEST", "Dim_Klant", "CHECK", 1,
    "expected", "actual", "SUCCESS", "test")

pd.DataFrame(log_list)


,run_id,source,table,action,key,expected,actual,status,note
0,20260412_204303,SDM → DWH,Dim_Monteur,SCD1 SUMMARY,-,Batch processed,"Inserted: 0, Updated: 15",SUCCESS,SCD1 load completed
1,20260412_204303,SDM → DWH,Dim_Klant,SCD2 SUMMARY,-,Batch processed,"Inserted: 0, Updated: 0, No change: 26",SUCCESS,SCD2 load completed
2,20260412_204303,SDM → DWH,Dim_Fiets,SCD2 SUMMARY,-,Batch processed,"Inserted: 0, Updated: 0, No change: 75",SUCCESS,SCD2 load completed
3,20260412_204303,SDM → DWH,Fact_Fiets_Verkoop,FACT LOAD,-,New facts inserted,0 rows inserted,SUCCESS,Incremental fact load
4,20260412_204303,TEST,Dim_Klant,CHECK,1,expected,actual,SUCCESS,test


test klant

In [20]:
SDM.execute("""
INSERT INTO Fietsverkoop_Klant
(klantnr, naam, woonplaats, geslacht, geboortedatum)
VALUES (999, 'Bob', 'Den Haag', 'M', '2001-11-01')
""")
SDM.commit()


IntegrityError: UNIQUE constraint failed: Fietsverkoop_Klant.klantnr

In [ ]:
load_scd2(df_klant_clean, "Dim_Klant", "klantnr",
          ["naam", "woonplaats", "geslacht", "geboortedatum"])


In [22]:
SDM.execute("""
INSERT INTO Fietsverkoop_Fiets_Verkoop
(fiets_verkoopnr, datum, aantal, verkoopprijs, klant, fiets, monteur)
VALUES (9999, '2024-01-01', 2, 500, 1, 1, 1)
""")
SDM.commit()


IntegrityError: UNIQUE constraint failed: Fietsverkoop_Fiets_Verkoop.fiets_verkoopnr

In [21]:
df_test = df_klant_clean.copy()
df_test.loc[0, "geboortedatum"] = "INVALID_DATE"

load_scd2(df_test.head(1), "Dim_Klant", "klantnr",
          ["naam", "woonplaats", "geslacht", "geboortedatum"])
